# 05 · Cross-Persona Convergence Analysis

Integrates outputs from notebooks 02–04 to produce a unified view of how much personas **converge** across three dimensions:

| Dimension | Measure |
|-----------|------|
| **Label** | Pairwise label agreement (Jaccard on perception tags) per image |
| **Sentiment** | Pairwise agreement on `predicted_sentiment` per image |
| **Caption text** | Mean cosine similarity of caption embeddings per image |

**Goal**: Identify which dimension shows the most divergence, and whether persona characteristics (demographics) explain divergence patterns.

In [1]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from scipy import stats

from src.config import FIGURES, OUTPUTS, FONT_SCALE, SENT_COLORS, SENT_COLORS_3
from src.data_loading import load_annotations, parse_demographics, create_profiles
from src.convergence import compute_sentiment_agreement, compute_label_jaccard, merge_convergence_dimensions

MUTED = sns.color_palette("muted")
sns.set_theme(style="whitegrid", font_scale=FONT_SCALE)
plt.rcParams["figure.dpi"] = 150

FIGURES.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

df = load_annotations()
df = parse_demographics(df)
df = create_profiles(df)
print(f"Records: {len(df):,} | Images: {df['image_id'].nunique():,} | Personas: {df['persona_id'].nunique():,}")

Records: 59,708 | Images: 50 | Personas: 1,200


## Correlation: justification cosine vs. perception Jaccard

In [2]:
from scipy.stats import pearsonr as _pearsonr
_just_mat    = np.load(OUTPUTS / "ic_profile_sim_just.npy")
_jaccard_mat = np.load(OUTPUTS / "ic_profile_sim_jaccard.npy")
_n = _just_mat.shape[0]
_idx = np.triu_indices(_n, k=1)
_r, _p = _pearsonr(_just_mat[_idx], _jaccard_mat[_idx])
print(f"Pearson r(justification cosine, Jaccard) = {_r:.4f}  "
      f"(p = {_p:.3e})  n = {len(_just_mat[_idx])} off-diagonal pairs")


Pearson r(justification cosine, Jaccard) = 0.6717  (p = 1.416e-37)  n = 276 off-diagonal pairs


## 1 · Load pre-computed similarity from notebook 03

In [3]:
sim_df = pd.read_csv(OUTPUTS / "per_image_cosine_similarity.csv")
sim_df = sim_df.rename(columns={"mean_sim": "caption_sim"})
print(f"Image-level caption similarity: {len(sim_df):,} images")

Image-level caption similarity: 50 images


## 2 · Compute per-image sentiment agreement

In [4]:
sent_agree_df = compute_sentiment_agreement(df)
print(f"Sentiment agreement computed for {len(sent_agree_df):,} images")
sent_agree_df.describe().round(4)

Sentiment agreement computed for 50 images


,sentiment_agreement
count,50.0000
mean,0.7589
std,0.2515
min,0.3163
25%,0.5122
50%,0.8772
75%,0.9913
max,1.0000


## 3 · Compute per-image perception label agreement (Jaccard)

In [5]:
label_agree_df = compute_label_jaccard(df)
print(f"Label Jaccard computed for {len(label_agree_df):,} images")
label_agree_df.describe().round(4)

Label Jaccard computed for 50 images


,label_jaccard
count,50.0000
mean,0.4370
std,0.1501
min,0.1407
25%,0.3443
50%,0.4170
75%,0.5235
max,0.8217


## 4 · Merge all three dimensions

In [6]:
conv_df = (
    sim_df[["image_id", "caption_sim", "n_personas"]]
    .merge(sent_agree_df[["image_id", "sentiment_agreement", "majority_sentiment"]], on="image_id")
    .merge(label_agree_df, on="image_id")
)
conv_df.to_csv(OUTPUTS / "convergence_all_dimensions.csv", index=False)
print(f"Convergence dataframe: {conv_df.shape}")
conv_df.describe().round(4)

Convergence dataframe: (50, 6)


,caption_sim,n_personas,sentiment_agreement,label_jaccard
count,50.0000,50.0000,50.0000,50.0000
mean,0.8727,1194.1600,0.7589,0.4370
std,0.0734,26.8939,0.2515,0.1501
min,0.6337,1015.0000,0.3163,0.1407
25%,0.8356,1199.0000,0.5122,0.3443
50%,0.8753,1200.0000,0.8772,0.4170
75%,0.9250,1200.0000,0.9913,0.5235
max,0.9798,1200.0000,1.0000,0.8217


## 8 · Kruskal-Wallis test across sentiment groups

In [7]:
order = ["Positive", "Neutral", "Negative"]
sub = conv_df[conv_df["majority_sentiment"].isin(order)]

results = []
for col in ["caption_sim", "sentiment_agreement", "label_jaccard"]:
    groups = [sub[sub["majority_sentiment"] == s][col].dropna() for s in order]
    stat, p = stats.kruskal(*groups)
    results.append({"dimension": col, "H_statistic": stat, "p_value": p})
    print(f"{col}: H={stat:.4f}, p={p:.4e}")

pd.DataFrame(results).to_csv(OUTPUTS / "kruskal_wallis_convergence.csv", index=False)

caption_sim: H=1.8957, p=3.8758e-01
sentiment_agreement: H=17.7556, p=1.3945e-04
label_jaccard: H=7.0653, p=2.9227e-02


## 9 · Add justification similarity as a 4th convergence dimension

Notebook 03 (sections 8–9) now computes pairwise cosine similarity of *justification*
embeddings per image. Including this dimension shows whether persona-driven text content
(captions + justifications) diverges or converges similarly to structured outputs
(sentiment labels and perception tags).

In [8]:
JUST_SIM_PATH = OUTPUTS / "per_image_just_similarity.csv"

if JUST_SIM_PATH.exists():
    just_sim_df = pd.read_csv(JUST_SIM_PATH)
    just_sim_df = just_sim_df.rename(columns={"mean_just_sim": "just_sim"})
    print(f"Loaded justification similarity: {len(just_sim_df):,} images")
    just_sim_df.describe().round(4)
else:
    print("Run notebook 03 sections 8+ first to generate per_image_just_similarity.csv")
    just_sim_df = None


Loaded justification similarity: 50 images


In [9]:
if just_sim_df is not None:
    conv4_df = conv_df.merge(
        just_sim_df[["image_id", "just_sim"]], on="image_id", how="left"
    )
    conv4_df.to_csv(OUTPUTS / "convergence_all_dimensions_4d.csv", index=False)
    print("4-dimension convergence dataframe:")
    print(conv4_df[["caption_sim", "sentiment_agreement", "label_jaccard", "just_sim"]].describe().round(4))


4-dimension convergence dataframe:
       caption_sim  sentiment_agreement  label_jaccard  just_sim
count      50.0000              50.0000        50.0000   50.0000
mean        0.8727               0.7589         0.4370    0.5639
std         0.0734               0.2515         0.1501    0.0611
min         0.6337               0.3163         0.1407    0.4192
25%         0.8356               0.5122         0.3443    0.5261
50%         0.8753               0.8772         0.4170    0.5558
75%         0.9250               0.9913         0.5235    0.6106
max         0.9798               1.0000         0.8217    0.7509
